**Tokenize/Encode Reviews - Transformer Models**

This notebook will repeat the encoding done in the Word2Vec notebook but across 3 different transformer models that represent different tradeoffs in computational time/cost and encoding quality. The approaches are centered around SBERT models, which are BERT variants trained with an emphasis on CLS embedding performance. This means that they are suited to summarizing sentences or passages with a single vector, which corresponds to this prediction task. All of the chosen models output 384 dimension embeddings. 768 dimensions is generally being avoided due to increased risk of overfitting in the FFNN prediction head with our somewhat limited dataset. It is also likely unnecessary given the focused semantic scope of the input text. All models will almost certainly perform better than the Word2Vec approach, which lacks context in its embeddings. The models are as follows:



all‑MiniLM‑L6‑v2 - Sbert model with 6 layers, trained on sentence level embeddings. sbert is specifically intended and fine tuned to generate quality embeddings to describe an entire sentence or passage in that one embedding. Context window is capped at 512 tokens which will truncate some reviews (affects ~4% of reviews on the user tower and ~2% on the business tower).

msmarco‑MiniLM‑L12‑v3 - larger sbert model with 12 layers, same general strengths as above but can express more attention based nuance. It is also trained for retrieval as opposed to being general purpose like the model above. The hypothesis is that this will likely perform better than the starting transformer.

jinaai/jina-embeddings-v2-base-en - This model is BERT-based but uses AliBi for it's positional encoding. ALiBi computes attention based on distances between tokens as opposed to absolute embeddings in vanilla BERT. The main benefit is that AliBi lets it take up to 8192 tokens as input and the model is also optimized specifically for retrieval. We will be reducing the window to 2048 tokens to be more closely aligned with our actual needs, with some built-in margin. As a small bookkeeping detail, it does require us to manually pool the output unlike the SBERT models. The output is also 768 dimensions, so we are at higher risk of overfitting as a tradeoff to the increased context length. And of course the higher context window will require extra computational power.

Install Libraries if needed, Imports as well

In [ ]:
pip install -U "transformers<5.0.0" sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.2/470.2 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 128.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 108.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Unins

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
from sentence_transformers import SentenceTransformer, models
import numpy as np
from tqdm import tqdm
import time
import torch
import numpy as np

SBERT 6 Layer (all-MiniLM-L6-v2)

In [ ]:
# Load SBERT model (all-MiniLM-L6-v2) — outputs 384-d embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

# Function to generate embeddings
def process_review_sbert_6layer(input_path, output_path, categories=False, batch_size=64):
    """
    Reads JSONL reviews, embeds either 'text' or 'categories' using all-MiniLM-L6-v2,
    replaces that field with the 384-d embedding (rounded to 5 decimals), and writes out new JSON.
    """
    with open(input_path, 'r', encoding='utf-8') as fin, \
         open(output_path, 'w', encoding='utf-8') as fout:

        buffer = []
        original_reviews = []

        for line in fin:
            review = json.loads(line)
            if categories:
                raw = review.get('categories', '')
            else:
                raw = review.get('text', '')
            # Ensure string
            if not isinstance(raw, str):
                raw = str(raw)
            buffer.append(raw)
            original_reviews.append(review)

            # When buffer fills, process batch
            if len(buffer) >= batch_size:
                embeddings = model.encode(buffer, convert_to_numpy=True, show_progress_bar=False)
                for rev, emb in zip(original_reviews, embeddings):
                    emb_rounded = [round(float(x), 4) for x in emb]
                    if categories:
                        rev['text'] = emb_rounded  # Save as "text" to keep consistent between files
                        rev.pop('categories', None)
                    else:
                        rev['text'] = emb_rounded
                    fout.write(json.dumps(rev) + '\n')
                buffer = []
                original_reviews = []

        # leftover reviews
        if buffer:
            embeddings = model.encode(buffer, convert_to_numpy=True, show_progress_bar=False)
            for rev, emb in zip(original_reviews, embeddings):
                emb_rounded = [round(float(x), 4) for x in emb]
                if categories:
                    rev['text'] = emb_rounded
                    rev.pop('categories', None)
                else:
                    rev['text'] = emb_rounded
                fout.write(json.dumps(rev) + '\n')

    print("Embeddings saved to:", output_path)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# File paths (adjust as needed)
INPUT_PATH  = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/Raw/user_tower_reviews.json'
OUTPUT_PATH = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/SBERT_6Layer/user_tower_reviews.json'

# Run the processing on each review file and category tags
start_time = time.time()
process_review_sbert_6layer(input_path = INPUT_PATH, output_path = OUTPUT_PATH)
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Embeddings saved to: /content/drive/MyDrive/Colab_Folder/266_Project/Data/SBERT_6Layer/user_tower_reviews.json
Time taken: 80.15586590766907 seconds


In [ ]:
# File paths (adjust as needed)
INPUT_PATH  = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/Raw/business_category_tags.json'
OUTPUT_PATH = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/SBERT_6Layer/business_category_tags.json'

# Run the processing on each review file and category tags
start_time = time.time()
process_review_sbert_6layer(input_path = INPUT_PATH, output_path = OUTPUT_PATH, categories=True)
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")

Embeddings saved to: /content/drive/MyDrive/Colab_Folder/266_Project/Data/SBERT_6Layer/business_category_tags.json
Time taken: 34.18895387649536 seconds


SBERT 12 Layer (msmarco‑MiniLM‑L12‑v3)

In [ ]:
# Load SBERT model (msmarco-MiniLM-L12-v3) — outputs 384-d embeddings
model = SentenceTransformer('sentence-transformers/msmarco-MiniLM-L12-v3')

# Function to generate embeddings
def process_review_sbert_12layer(input_path, output_path, categories=False, batch_size=64):
    """
    Reads JSONL reviews, embeds either 'text' or 'categories' using msmarco-MiniLM-L12-v3,
    replaces that field with the 384-d embedding (rounded to 5 decimals), and writes out new JSON.
    """
    with open(input_path, 'r', encoding='utf-8') as fin, \
         open(output_path, 'w', encoding='utf-8') as fout:

        buffer = []
        original_reviews = []

        for line in fin:
            review = json.loads(line)
            if categories:
                raw = review.get('categories', '')
            else:
                raw = review.get('text', '')
            # Ensure string
            if not isinstance(raw, str):
                raw = str(raw)
            buffer.append(raw)
            original_reviews.append(review)

            # When buffer fills, process batch
            if len(buffer) >= batch_size:
                embeddings = model.encode(buffer, convert_to_numpy=True, show_progress_bar=False)
                for rev, emb in zip(original_reviews, embeddings):
                    emb_rounded = [round(float(x), 4) for x in emb]
                    if categories:
                        rev['text'] = emb_rounded  # Save as "text" to keep consistent between files
                        rev.pop('categories', None)
                    else:
                        rev['text'] = emb_rounded
                    fout.write(json.dumps(rev) + '\n')
                buffer = []
                original_reviews = []

        # leftover reviews
        if buffer:
            embeddings = model.encode(buffer, convert_to_numpy=True, show_progress_bar=False)
            for rev, emb in zip(original_reviews, embeddings):
                emb_rounded = [round(float(x), 4) for x in emb]
                if categories:
                    rev['text'] = emb_rounded
                    rev.pop('categories', None)
                else:
                    rev['text'] = emb_rounded
                fout.write(json.dumps(rev) + '\n')

    print("Embeddings saved to:", output_path)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# File paths (adjust as needed)
INPUT_PATH  = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/Raw/user_tower_reviews.json'
OUTPUT_PATH = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/SBERT_12Layer/user_tower_reviews.json'

# Run the processing on each review file and category tags
start_time = time.time()
process_review_sbert_12layer(input_path = INPUT_PATH, output_path = OUTPUT_PATH)
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")

Embeddings saved to: /content/drive/MyDrive/Colab_Folder/266_Project/Data/SBERT_12Layer/user_tower_reviews.json
Time taken: 142.25191450119019 seconds


In [ ]:
# File paths (adjust as needed)
INPUT_PATH  = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/Raw/business_category_tags.json'
OUTPUT_PATH = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/SBERT_12Layer/business_category_tags.json'

# Run the processing on each review file and category tags
start_time = time.time()
process_review_sbert_12layer(input_path = INPUT_PATH, output_path = OUTPUT_PATH, categories=True)
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")

Embeddings saved to: /content/drive/MyDrive/Colab_Folder/266_Project/Data/SBERT_12Layer/business_category_tags.json
Time taken: 41.73443603515625 seconds


jinaai/jina-embeddings-v2-base-en

In [ ]:
# Load JinaAI long-context 768-d encoder
MODEL_NAME = "jinaai/jina-embeddings-v2-base-en"
model = SentenceTransformer(MODEL_NAME)
model.max_seq_length = 2048 # Set context size


def process_review_jina(input_path, output_path, categories=False, batch_size=64):
    with open(input_path, 'r', encoding='utf-8') as fin, \
         open(output_path, 'w', encoding='utf-8') as fout:

        buffer = []
        original_reviews = []

        for line in fin:
            review = json.loads(line)
            if categories:
                raw = review.get('categories', '')
            else:
                raw = review.get('text', '')
            if not isinstance(raw, str):
                raw = str(raw)
            buffer.append(raw)
            original_reviews.append(review)

            if len(buffer) >= batch_size:
                embeddings = model.encode(buffer, convert_to_numpy=True, show_progress_bar=False)
                for rev, emb in zip(original_reviews, embeddings):
                    emb_rounded = [round(float(x), 4) for x in emb]
                    rev['text'] = emb_rounded
                    if categories:
                        rev.pop('categories', None)
                    fout.write(json.dumps(rev) + '\n')
                buffer = []
                original_reviews = []

        if buffer:
            embeddings = model.encode(buffer, convert_to_numpy=True, show_progress_bar=False)
            for rev, emb in zip(original_reviews, embeddings):
                emb_rounded = [round(float(x), 4) for x in emb]
                rev['text'] = emb_rounded
                if categories:
                    rev.pop('categories', None)
                fout.write(json.dumps(rev) + '\n')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Some weights of BertModel were not initialized from the model checkpoint at jinaai/jina-embeddings-v2-base-en and are newly initialized: ['embeddings.position_embeddings.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.dense.bias', 'encoder.layer.0.output.dense.weight', 'encoder.layer.1.intermediate.dense.bias', 'encoder.layer.1.intermediate.dense.weight', 'encoder.layer.1.output.LayerNorm.bias', 'encoder.layer.1.output.LayerNorm.weight', 'encoder.layer.1.output.dense.bias', 'encoder.layer.1.output.dense.weight', 'encoder.layer.10.intermediate.dense.bias', 'encoder.layer.10.intermediate.dense.weight', 'encoder.layer.10.output.LayerNorm.bias', 'encoder.layer.10.output.LayerNorm.weight', 'encoder.layer.10.output.dense.bias', 'encoder.layer.10.output.dense.weight', 'encoder.layer.11.intermediate.dense.bias', 'encoder.layer.11.intermedi

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# File paths (adjust as needed)
INPUT_PATH  = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/Raw/user_tower_reviews.json'
OUTPUT_PATH = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/JINA/user_tower_reviews.json'

# Run the processing on each review file and category tags
start_time = time.time()
process_review_jina(input_path = INPUT_PATH, output_path = OUTPUT_PATH)
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")

Time taken: 544.4319009780884 seconds


In [ ]:
# File paths (adjust as needed)
INPUT_PATH  = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/Raw/business_category_tags.json'
OUTPUT_PATH = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/JINA/business_category_tags.json'

# Run the processing on each review file and category tags
start_time = time.time()
process_review_jina(input_path = INPUT_PATH, output_path = OUTPUT_PATH, categories=True)
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")

Time taken: 70.19371151924133 seconds


Run Notes:

A T4 GPU and high RAM instance crashed on the JINA model, showing how costly it is to run. We will move up to L4 with high RAM and see how it goes.

L4 also crashed with JINA. Moving up to A100 for this.

For reference, CPU only with high RAM (word2vec) is around 0.22 units an hour. L4 with high RAM is around 1.9 units an hour. 10 unit corresponds to 1 dollar on colab. A100 with high RAM is around 6.00 an hour. 1 unit corresponds to 10 cents at the time of execution.
